# Phase 6 · Submission S1 — Architecture A (Decoder-only Transformer) + LORMA, chunk-level abstractive summarization

**Task:** abstractive summarization of long legal judgments. **Primary metric:** BERTScore F1 (secondary ROUGE-1 F1, ROUGE-L F1).

**What the data analysis showed** (`P6_data_analysis`): reference summaries are long (median 745 BPE tokens ≈ 22% of the document), their content comes **evenly from the whole document**, they paraphrase document sentences closely, and only 19% of documents fit in A's 2,048-token context.

**Method — chunk-level summaries joined into a document summary**
1. **Chunks:** each document is split into sentences and grouped into consecutive chunks of ≤ 1,200 BPE tokens (no sentence is cut unless it alone is longer).
2. **Aligned training pairs:** every reference-summary sentence is assigned to the chunk that contains its best-matching document sentence (highest word recall); the chunk's target is its assigned summary sentences in summary order. ≈ 7,000 documents → tens of thousands of (chunk → partial summary) pairs.
3. **Model:** Phase-1 Architecture A as a prefix language model: `[BOS] chunk " Summary:" target [EOS]` (≤ 2,048 tokens); the loss is computed on the target tokens only. The tied Phase-1 output layer is the generator, so nothing is initialised at random.
4. **Adaptation — LORMA (compulsory):** q, k, v, out, fc1, fc2 of all 12 layers get `W_eff = (I + (α/r)·B·A)·W0` (r = 8, α = 16, `B = 0` at start); everything else frozen; dropout off.
5. **Generation:** every chunk of a document is summarised with cached (KV) greedy decoding, repeated 4-grams blocked, a minimum / maximum length proportional to the chunk; the pieces are joined in document order and sentences already produced are dropped (redundancy reduction).
6. **Validation:** 150 training documents are held out (never trained on). Epoch selection by validation loss; ROUGE-1 / ROUGE-L of full generated summaries on the held-out documents; the share of each chunk summary that is kept (length control) is tuned there.
7. **Long-document comparison for the report:** all chunks vs the first chunk only (≈ what fits in the context window).
8. **For S4 (knowledge distillation):** the trained LORMA weights (`checkpoints/best.pt`) and the exact training pairs (`data_cache.pt`) are saved.

**Before running (Kaggle)**
- Accelerator: **GPU T4 ×2** · Add-ons → Secrets: **`HF`** · Internet on.
- Add Input → **`rohitbatra2004/phase111`** (Architecture A) and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- Expected time ≈ 4–5 hours (data preparation ≈ 10 min, ≈ 70–90 min per epoch, generation on validation + test ≈ 30–45 min). Checkpoints every 1,000 steps and every epoch; the prepared data is cached. To resume, add this notebook's previous output as an input and run again.

Outputs: `/kaggle/working/P6_S1_A_LORMA/` — upload `submission.zip`; then save this output folder as a Kaggle dataset (`Phase6_S1_Dataset`, the teacher for S4).

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib, re
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P6_S1_A_LORMA"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Chunk-level summarization data.
CHUNK_TOKENS = 1200                 # document tokens per chunk (whole sentences)
MAX_TARGET_TOKENS = 640             # partial-summary tokens per chunk used in training (longer targets are cut)
MAX_CHUNKS_PER_DOC = 40             # chunks kept per document (> 99.5% of documents complete)
SEPARATOR = " Summary:"             # text between the chunk and its summary
VAL_DOCS = 150                      # held-out training documents for validation (never trained on)

# LORMA on attention and feed-forward projections of all 12 layers.
LORMA_RANK = 8
LORMA_ALPHA = 16.0
LORMA_TARGETS_ATTN = ("q_proj", "k_proj", "v_proj", "out_proj")
LORMA_TARGETS_FFN = ("fc1", "fc2")

# Training.
EPOCHS = 2
BATCH_SEQS = 4                      # sequences (≤ 2,048 tokens) per forward pass, split over both GPUs
GRAD_ACCUM = 4                      # effective batch = 16 sequences
LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.03
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 1000

# Generation (greedy, cached).
GEN_BATCH = 16
MIN_NEW_FRACTION = 0.08             # minimum summary tokens per chunk = 8% of the chunk's tokens …
MAX_NEW_FRACTION = 0.45             # … maximum = 45% of the chunk's tokens (+ 32), capped at MAX_TARGET_TOKENS
NO_REPEAT_NGRAM = 4
KEEP_FRACTIONS = (0.5, 0.7, 0.85, 1.0)   # share of each chunk summary's sentences kept (tuned on validation)
USE_AMP = DEVICE.type == "cuda"
print("Run:", RUN_NAME, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download Phase_6_Summ from Hugging Face using the Kaggle secret "HF".
from kaggle_secrets import UserSecretsClient
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_6_Summ/**"])
P6_DIR = DATA_DIR / "Phase_6_Summ"
for name in ("Train/train.json", "Test/test.json"):
    assert (P6_DIR / name).is_file(), f"Missing {name}"
print("Phase 6 data ready in", P6_DIR)

In [ ]:
# Cell 4 — Load train / test, drop empty summaries and duplicate documents, and hold out VAL_DOCS training documents.
def clean(text):
    return " ".join(text.split())

raw_train = json.load(open(P6_DIR / "Train/train.json"))
raw_test = json.load(open(P6_DIR / "Test/test.json"))
docs, seen = [], set()
for d in raw_train:
    document, summary = clean(d["document"]), clean(d["summary"])
    key = hashlib.md5(document.encode()).hexdigest()
    if not summary or key in seen:
        continue
    seen.add(key)
    docs.append({"id": str(d["id"]), "document": document, "summary": summary})
order = list(range(len(docs)))
random.Random(SEED).shuffle(order)
val_docs = [docs[i] for i in order[:VAL_DOCS]]
train_docs = [docs[i] for i in order[VAL_DOCS:]]
test_docs = [{"id": str(d["id"]), "document": clean(d["document"])} for d in raw_test]
del raw_train, raw_test
print(f"train {len(train_docs):,} documents | validation {len(val_docs)} (held out) | test {len(test_docs)}")

In [ ]:
# Cell 5 — Sentence-aligned chunks and (chunk → partial summary) pairs: each summary sentence goes to the chunk holding its
# best-matching document sentence (highest word recall). Cached in data_cache.pt (also used by S4).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]", "[UNK]"))
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID) == (0, 1, 2, 3)
SEP_IDS = tokenizer.encode(SEPARATOR).ids
SENT_SPLIT = re.compile(r"(?<=[.?!])\s+(?=[A-Z0-9(\"'])")
WORD = re.compile(r"[a-z0-9]+")

def split_sentences(text):
    return [s for s in SENT_SPLIT.split(text) if s.strip()]

def encode_sentences(sentences):
    return [np.array(e.ids, dtype=np.uint16) for e in tokenizer.encode_batch([" " + s for s in sentences])] if sentences else []

def make_chunks(doc_text):
    """Chunks of whole sentences (≤ CHUNK_TOKENS tokens) and, for every document sentence, the chunk it lies in."""
    sentences = split_sentences(doc_text)
    chunks, sentence_chunk, current = [], [], []
    for ids in encode_sentences(sentences):
        pieces = [ids[i:i + CHUNK_TOKENS] for i in range(0, max(len(ids), 1), CHUNK_TOKENS)]
        sentence_chunk.append(None)
        for piece in pieces:
            if current and sum(len(p) for p in current) + len(piece) > CHUNK_TOKENS:
                chunks.append(np.concatenate(current)); current = []
            if sentence_chunk[-1] is None:
                sentence_chunk[-1] = len(chunks)
            current.append(piece)
    if current:
        chunks.append(np.concatenate(current))
    return sentences, chunks[:MAX_CHUNKS_PER_DOC], sentence_chunk

def aligned_targets(doc_sentences, sentence_chunk, n_chunks, summary):
    """Partial summary (token array) per chunk and the word recall of every alignment."""
    doc_sets = [set(WORD.findall(s.lower())) for s in doc_sentences]
    parts, recalls = [[] for _ in range(n_chunks)], []
    summary_sentences = split_sentences(summary)
    for sent, ids in zip(summary_sentences, encode_sentences(summary_sentences)):
        words = set(WORD.findall(sent.lower()))
        if not words or not doc_sets:
            continue
        scores = [len(words & ds) / len(words) for ds in doc_sets]
        best = int(np.argmax(scores))
        recalls.append(scores[best])
        chunk = min(sentence_chunk[best], n_chunks - 1)
        parts[chunk].append(ids)
    return [np.concatenate(p)[:MAX_TARGET_TOKENS] if p else np.zeros(0, dtype=np.uint16) for p in parts], recalls

def prepare(doc_list, with_targets):
    out, recalls = [], []
    for d in doc_list:
        sentences, chunks, sentence_chunk = make_chunks(d["document"])
        if not chunks:
            chunks, sentence_chunk = [np.array(tokenizer.encode(" " + d["document"]).ids[:CHUNK_TOKENS], dtype=np.uint16)], [0]
        item = {"id": d["id"], "chunks": chunks}
        if with_targets:
            item["targets"], r = aligned_targets(sentences, sentence_chunk, len(chunks), d["summary"])
            item["summary"] = d["summary"]; recalls += r
        out.append(item)
    return out, recalls

cache = WORK_DIR / "data_cache.pt"
if not cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/data_cache.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], cache)
if cache.exists():
    data = torch.load(cache, weights_only=False)
    print("Loaded prepared data from", cache)
else:
    start_time = time.time()
    train_items, recalls = prepare(train_docs, True)
    val_items, _ = prepare(val_docs, True)
    test_items, _ = prepare(test_docs, False)
    data = {"train": train_items, "val": val_items, "test": test_items, "alignment_recall": recalls,
            "settings": {"chunk_tokens": CHUNK_TOKENS, "max_target_tokens": MAX_TARGET_TOKENS, "separator": SEPARATOR}}
    torch.save(data, cache)
    print(f"Prepared chunks and aligned targets in {(time.time() - start_time) / 60:.1f} min → {cache}")
assert [d["id"] for d in data["val"]] == [d["id"] for d in val_docs] and [d["id"] for d in data["test"]] == [d["id"] for d in test_docs]

def pairs_of(items):
    return [(np.asarray(c), np.asarray(t)) for d in items for c, t in zip(d["chunks"], d["targets"]) if len(t) > 0]
train_pairs, val_pairs = pairs_of(data["train"]), pairs_of(data["val"])
chunk_counts = np.array([len(d["chunks"]) for d in data["train"]])
target_share = np.mean([len(t) / max(len(c), 1) for c, t in train_pairs])
recalls = np.array(data["alignment_recall"])
print(f"train pairs {len(train_pairs):,} | validation pairs {len(val_pairs):,} | chunks per document median "
      f"{int(np.median(chunk_counts))}, p90 {int(np.percentile(chunk_counts, 90))} | target ≈ {100 * target_share:.0f}% of its chunk")
print(f"summary sentences aligned with word recall ≥ 0.5: {100 * np.mean(recalls >= 0.5):.1f}% (median {np.median(recalls):.2f}) | "
      f"chunks with no summary sentence: {100 * np.mean([len(t) == 0 for d in data['train'] for t in d['targets']]):.1f}%")

In [ ]:
# Cell 6 — Phase-1 Architecture A classes and the pretrained weights, verified against the Phase-1 metadata.
# The weights stay in memory so that build_model() can create a fresh adapted copy.
a_paths = sorted(glob.glob("/kaggle/input/**/architecture_A/model_A_state_dict.pt", recursive=True))
assert a_paths, "Add the dataset rohitbatra2004/phase111 as an input."
A_ROOT = Path(a_paths[0]).parent
A_METADATA = json.load(open(A_ROOT / "model_A_metadata.json"))
A_CONFIG = A_METADATA["model_config"]
assert A_CONFIG["pre_norm"] and A_CONFIG["tie_embeddings"], "Phase-1 A is a pre-LN decoder with tied embeddings"

class CausalSelfAttention(nn.Module):
    """Same parameters as Phase 1; uses PyTorch's fused causal attention for speed and memory."""
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.n_heads, self.dropout = cfg["n_heads"], cfg["dropout"]
        self.q_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.k_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.v_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.out_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.resid_dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, T, C = x.shape
        split = lambda t: t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        out = F.scaled_dot_product_attention(split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x)),
                                             dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.resid_dropout(self.out_proj(out.transpose(1, 2).reshape(B, T, C)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["use_bias"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["use_bias"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["d_model"])
        self.attention = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["d_model"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        return x + self.feed_forward(self.ln_2(x))

class ArchitectureABackbone(nn.Module):
    """Phase-1 decoder-only LM without the vocabulary projection.
    Returns the final-normalised state of every token: [batch, tokens, d_model]."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(cfg["d_model"])
        # Tied output weight, kept so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_weight = self.token_embedding.weight

    def forward(self, input_ids):
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)

saved = torch.load(a_paths[0], map_location="cpu", weights_only=True)
# The export is either the raw state dict or a dict that wraps it under "state_dict".
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved

check = ArchitectureABackbone(A_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(A_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(A_CONFIG, indent=2))
print(f"Phase-1 Architecture A weights verified from {A_ROOT} — {n_backbone:,} parameters")

In [ ]:
# Cell 7 — LORMA: multiplicative low-rank update of a frozen linear layer, W_eff = (I + (α/r)·B·A)·W0.
# Applied to q/k/v/out and fc1/fc2 in all 12 layers; B = 0 at start, so the model initially equals Phase 1.
class LoRMALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lorma_A = nn.Parameter(torch.empty(rank, base.out_features))
        self.lorma_B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

    def forward(self, x):
        w0x = F.linear(x, self.base.weight)                               # W0 x (without bias)
        update = F.linear(F.linear(w0x, self.lorma_A), self.lorma_B)       # B (A (W0 x))
        out = w0x + self.scale * update
        return out + self.base.bias if self.base.bias is not None else out

def make_lorma_backbone():
    bb = ArchitectureABackbone(A_CONFIG)
    bb.load_state_dict(PHASE1_STATE, strict=True)
    for p in bb.parameters():
        p.requires_grad_(False)
    for block in bb.blocks:
        for name in LORMA_TARGETS_ATTN:
            setattr(block.attention, name, LoRMALinear(getattr(block.attention, name), LORMA_RANK, LORMA_ALPHA))
        for name in LORMA_TARGETS_FFN:
            setattr(block.feed_forward, name, LoRMALinear(getattr(block.feed_forward, name), LORMA_RANK, LORMA_ALPHA))
    return bb

# Checks on the device: (1) with B = 0 the output equals the frozen Phase-1 model; (2) A and B receive gradients.
probe_ids = torch.randint(4, 1000, (2, 16), device=DEVICE)
plain = ArchitectureABackbone(A_CONFIG); plain.load_state_dict(PHASE1_STATE, strict=True); plain = plain.to(DEVICE).eval()
probe_bb = make_lorma_backbone().to(DEVICE).eval()
with torch.no_grad():
    assert torch.allclose(plain(probe_ids), probe_bb(probe_ids), atol=1e-4), "LORMA with B = 0 must equal Phase 1"
    lorma_layers = [m for m in probe_bb.modules() if isinstance(m, LoRMALinear)]
    for m in lorma_layers:
        m.lorma_B.normal_(0, 1e-3)                                       # temporary non-zero B for the gradient check
probe_bb.train()
probe_bb(probe_ids).float().pow(2).mean().backward()
assert all(m.lorma_A.grad is not None and m.lorma_A.grad.abs().sum() > 0 for m in lorma_layers), "LORMA gets no gradient"
n_lorma = sum(p.numel() for p in probe_bb.parameters() if p.requires_grad)
print(f"LORMA on {len(lorma_layers)} layers — {n_lorma:,} trainable backbone parameters ({100 * n_lorma / n_backbone:.2f}%)")
del plain, probe_bb, lorma_layers
torch.cuda.empty_cache()

In [ ]:
# Cell 8 — Prefix language model: logits = final states × tied Phase-1 embedding; the loss covers the summary tokens only.
class SummaryLM(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, ids, labels):
        """ids, labels [B, T] (labels = next token, -100 where no loss) → (summed cross-entropy, number of target tokens)."""
        states = self.backbone(ids)
        keep = labels.ne(-100)
        logits = F.linear(states[keep], self.backbone.token_embedding.weight).float()
        return F.cross_entropy(logits, labels[keep], reduction="sum").unsqueeze(0), keep.sum().float().unsqueeze(0)

torch.manual_seed(SEED)
model = SummaryLM(make_lorma_backbone()).to(DEVICE)
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = list(TRAINABLE_NAMES)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
N_GPUS = torch.cuda.device_count()
train_model = nn.DataParallel(model) if N_GPUS > 1 else model
CONTEXT = A_CONFIG["context_length"]
assert CHUNK_TOKENS + len(SEP_IDS) + MAX_TARGET_TOKENS + 2 <= CONTEXT, "Chunk + summary must fit in A's context"
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%) | GPUs used: {max(N_GPUS, 1)} | "
      f"context {CONTEXT} tokens")

In [ ]:
# Cell 9 — Batches: sequences of similar length are grouped (less padding); labels = next token on the summary part only.
def make_batch(pairs):
    seqs, prompt_lens = [], []
    for chunk, target in pairs:
        prompt = [BOS_ID] + chunk.tolist() + SEP_IDS
        seqs.append(prompt + target.tolist() + [EOS_ID]); prompt_lens.append(len(prompt))
    T = max(len(s) for s in seqs) - 1
    ids = torch.full((len(seqs), T), PAD_ID, dtype=torch.long)
    labels = torch.full((len(seqs), T), -100, dtype=torch.long)
    for b, (s, p) in enumerate(zip(seqs, prompt_lens)):
        ids[b, :len(s) - 1] = torch.tensor(s[:-1])
        labels[b, p - 1:len(s) - 1] = torch.tensor(s[p:])          # predict every summary token and the final [EOS]
    return ids.to(DEVICE), labels.to(DEVICE)

def length_batches(pairs, batch_size, seed):
    order = sorted(range(len(pairs)), key=lambda i: len(pairs[i][0]) + len(pairs[i][1]))
    batches = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    random.Random(seed).shuffle(batches)
    return batches

@torch.no_grad()
def validation_loss(net):
    net.eval()
    total, count = 0.0, 0.0
    for batch in length_batches(val_pairs, BATCH_SEQS * 2, SEED):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss_sum, n = net(*make_batch([val_pairs[i] for i in batch]))
        total, count = total + loss_sum.sum().item(), count + n.sum().item()
    return total / max(count, 1)

ids, labels = make_batch(train_pairs[:2])
print("Example batch:", tuple(ids.shape), "| target tokens:", int(labels.ne(-100).sum()),
      f"| {math.ceil(len(train_pairs) / BATCH_SEQS):,} batches per epoch")

In [ ]:
# Cell 10 — Cached (KV) batched greedy generation for the adapted decoder, with repeated 4-grams blocked and min / max lengths.
# Prompts are left-padded; padding keys are masked and positions start at each prompt's first real token.
def attend(attn, h, key_mask, cache):
    B, T, Cd = h.shape
    split = lambda t: t.view(B, T, attn.n_heads, Cd // attn.n_heads).transpose(1, 2)
    q, k, v = split(attn.q_proj(h)), split(attn.k_proj(h)), split(attn.v_proj(h))
    if cache is not None:
        k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
    S = k.size(2)
    mask = key_mask[:, None, None, :S].expand(B, 1, T, S)
    if T > 1:                                                     # prefill: causal + padding mask (a query always sees itself)
        causal = torch.ones(T, S, dtype=torch.bool, device=h.device).tril(S - T)
        mask = (mask & causal) | torch.eye(T, S, dtype=torch.bool, device=h.device)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    return attn.out_proj(out.transpose(1, 2).reshape(B, T, Cd)), (k, v)

def run_layers(bb, x, key_mask, caches):
    new = []
    for block, cache in zip(bb.blocks, caches):
        a, kv = attend(block.attention, block.ln_1(x), key_mask, cache)
        x = x + a
        x = x + block.feed_forward(block.ln_2(x))
        new.append(kv)
    return bb.final_norm(x), new

def block_repeats(logits, generated, n):
    """Forbid tokens that would repeat an n-gram already generated in that row."""
    for b, seq in enumerate(generated):
        if len(seq) >= n - 1:
            prefix = tuple(seq[-(n - 1):])
            banned = [seq[i + n - 1] for i in range(len(seq) - n + 1) if tuple(seq[i:i + n - 1]) == prefix]
            if banned:
                logits[b, banned] = -1e4

@torch.no_grad()
def generate(bb, prompts, max_new, min_new):
    """prompts: token lists; max_new / min_new: per-prompt limits → generated token lists (without [EOS])."""
    bb.eval()
    B, L = len(prompts), max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=DEVICE)
    for b, p in enumerate(prompts):
        ids[b, L - len(p):] = torch.tensor(p, device=DEVICE)
    key_mask = ids.ne(PAD_ID)
    positions = (key_mask.long().cumsum(1) - 1).clamp(min=0)
    weight = bb.token_embedding.weight
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        x = bb.token_embedding(ids) + bb.position_embedding(positions)
        states, caches = run_layers(bb, x, key_mask, [None] * len(bb.blocks))
        next_pos = positions[:, -1] + 1
        generated, done = [[] for _ in range(B)], [False] * B
        logits = F.linear(states[:, -1], weight).float()
        for step in range(max(max_new)):
            logits[:, [PAD_ID, BOS_ID, UNK_ID]] = -1e4
            for b in range(B):
                if step < min_new[b]:
                    logits[b, EOS_ID] = -1e4
            block_repeats(logits, generated, NO_REPEAT_NGRAM)
            token = logits.argmax(-1)
            for b in range(B):
                if not done[b]:
                    if token[b].item() == EOS_ID or len(generated[b]) >= max_new[b]:
                        done[b] = True
                    else:
                        generated[b].append(token[b].item())
            if all(done):
                break
            key_mask = torch.cat([key_mask, torch.ones(B, 1, dtype=torch.bool, device=DEVICE)], 1)
            x = bb.token_embedding(token[:, None]) + bb.position_embedding(next_pos.clamp(max=CONTEXT - 1)[:, None])
            states, caches = run_layers(bb, x, key_mask, caches)
            next_pos = next_pos + 1
            logits = F.linear(states[:, -1], weight).float()
    return generated

def summarise(items, first_chunk_only=False):
    """Per document: list of per-chunk summaries (each a list of sentences)."""
    jobs = [(i, j, c) for i, d in enumerate(items) for j, c in enumerate(d["chunks"][:1] if first_chunk_only else d["chunks"])]
    jobs.sort(key=lambda job: len(job[2]))
    out = [dict() for _ in items]
    for s in range(0, len(jobs), GEN_BATCH):
        batch = jobs[s:s + GEN_BATCH]
        prompts = [[BOS_ID] + c.tolist() + SEP_IDS for _, _, c in batch]
        max_new = [min(MAX_TARGET_TOKENS, int(MAX_NEW_FRACTION * len(c)) + 32) for _, _, c in batch]
        min_new = [int(MIN_NEW_FRACTION * len(c)) for _, _, c in batch]
        for (i, j, _), g in zip(batch, generate(model.backbone, prompts, max_new, min_new)):
            out[i][j] = split_sentences(tokenizer.decode(g).strip())
    return [[out[i][j] for j in sorted(out[i])] for i in range(len(items))]

def join_summary(chunk_summaries, keep_fraction=1.0):
    """Keep the first share of each chunk summary's sentences, drop sentences already used, join in document order."""
    seen, kept = set(), []
    for sentences in chunk_summaries:
        for s in sentences[:max(1, math.ceil(keep_fraction * len(sentences)))] if sentences else []:
            key = " ".join(WORD.findall(s.lower()))
            if key and key not in seen:
                seen.add(key); kept.append(s)
    return " ".join(kept)
print("Generation helpers ready.")

In [ ]:
# Cell 11 — ROUGE-1 / ROUGE-L F1 (word level, lower-cased) and repetition, for the held-out validation documents.
def rouge1_f(ref, hyp):
    overlap = sum((Counter(ref) & Counter(hyp)).values())
    return 0.0 if overlap == 0 else 2 * overlap / (len(ref) + len(hyp))

def rougeL_f(ref, hyp, cap=800):
    ref, hyp = ref[:cap], hyp[:cap]
    prev = [0] * (len(hyp) + 1)
    for x in ref:
        cur = [0]
        for j, y in enumerate(hyp, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return 0.0 if prev[-1] == 0 else 2 * prev[-1] / (len(ref) + len(hyp))

def score_summaries(refs, hyps):
    r1, rl, lengths, rep = [], [], [], []
    for ref, hyp in zip(refs, hyps):
        rw, hw = WORD.findall(ref.lower()), WORD.findall(hyp.lower())
        r1.append(rouge1_f(rw, hw)); rl.append(rougeL_f(rw, hw)); lengths.append(len(hw) / max(len(rw), 1))
        grams = list(zip(hw, hw[1:], hw[2:]))
        rep.append(1 - len(set(grams)) / max(len(grams), 1))
    return {"rouge1": float(np.mean(r1)), "rougeL": float(np.mean(rl)), "length_ratio": float(np.median(lengths)),
            "repeated_trigram_share": float(np.mean(rep))}

def describe(s):
    return (f"ROUGE-1 {s['rouge1']:.4f} | ROUGE-L {s['rougeL']:.4f} | length vs reference ×{s['length_ratio']:.2f} | "
            f"repeated trigrams {100 * s['repeated_trigram_share']:.1f}%")
print("Scoring ready (validation only).")

In [ ]:
# Cell 12 — Optimiser, learning-rate schedule and checkpoint resume (every 1,000 steps and every epoch).
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_pairs) / BATCH_SEQS)
total_updates = EPOCHS * math.ceil(steps_per_epoch / GRAD_ACCUM)
warmup_updates = int(WARMUP_FRACTION * total_updates)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda u: (u + 1) / max(warmup_updates, 1) if u < warmup_updates
                                              else max(0.0, (total_updates - u) / max(total_updates - warmup_updates, 1)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def trainable_state():
    full = model.state_dict()
    return {n: full[n].detach().cpu() for n in SAVED_NAMES}

LATEST, BEST = CKPT_DIR / "latest.pt", CKPT_DIR / "best.pt"
if not LATEST.exists():
    for previous in sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/latest.pt", recursive=True)):
        if set(torch.load(previous, map_location="cpu", weights_only=False)["trainable"]) == set(SAVED_NAMES):
            shutil.copy(previous, LATEST)
            if Path(previous).with_name("best.pt").exists():
                shutil.copy(Path(previous).with_name("best.pt"), BEST)
            print("Copied checkpoints from", Path(previous).parent)
            break

state = {"epoch": 0, "step": 0, "best_val_loss": float("inf"), "best_epoch": 0, "history": [], "train_seconds": 0.0}
if LATEST.exists():
    ckpt = torch.load(LATEST, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["trainable"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"]); scheduler.load_state_dict(ckpt["scheduler"]); scaler.load_state_dict(ckpt["scaler"])
    state = ckpt["state"]
    print(f"Resumed at epoch {state['epoch'] + 1}, step {state['step']}")
else:
    state["val_loss_phase1"] = validation_loss(train_model)        # epoch 0: the unadapted Phase-1 model
    print(f"No checkpoint found — training starts from the Phase-1 weights | validation loss before training "
          f"{state['val_loss_phase1']:.4f} (perplexity {math.exp(state['val_loss_phase1']):.1f})")
print(f"{steps_per_epoch:,} steps per epoch, {total_updates:,} optimiser updates in total")

In [ ]:
# Cell 13 — Training: next-token cross-entropy on the summary tokens, mixed precision, gradient accumulation, frozen
# backbone without dropout. Every epoch ends with the validation loss; best.pt keeps the epoch with the lowest one.
def save_latest():
    torch.save({"trainable": trainable_state(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}, LATEST)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
while state["epoch"] < EPOCHS:
    epoch = state["epoch"]
    batches = length_batches(train_pairs, BATCH_SEQS, SEED + epoch)
    train_model.train(); model.backbone.eval()
    start_time, loss_sum, token_sum = time.time(), 0.0, 0.0
    for step in range(state["step"], len(batches)):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            s, n = train_model(*make_batch([train_pairs[i] for i in batches[step]]))
        loss = s.sum() / n.sum().clamp(min=1)
        scaler.scale(loss / GRAD_ACCUM).backward()
        loss_sum, token_sum = loss_sum + s.sum().item(), token_sum + n.sum().item()
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(batches):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        state["step"] = step + 1
        if state["step"] % SAVE_EVERY_STEPS == 0:
            state["train_seconds"] += time.time() - start_time; start_time = time.time()
            save_latest()
            print(f"  epoch {epoch + 1} step {state['step']}/{len(batches)} | train loss {loss_sum / token_sum:.4f}", flush=True)
    state["train_seconds"] += time.time() - start_time
    val_loss = validation_loss(train_model)
    state["history"].append({"epoch": epoch + 1, "train_loss": loss_sum / max(token_sum, 1), "val_loss": val_loss})
    mark = ""
    if val_loss < state["best_val_loss"]:
        state.update(best_val_loss=val_loss, best_epoch=epoch + 1)
        torch.save({"trainable": trainable_state(), "epoch": epoch + 1, "val_loss": val_loss}, BEST)
        mark = "  <- best"
    state["epoch"], state["step"] = epoch + 1, 0
    save_latest()
    print(f"Epoch {epoch + 1} | train loss {loss_sum / max(token_sum, 1):.4f} | validation loss {val_loss:.4f} "
          f"(perplexity {math.exp(val_loss):.1f}){mark}", flush=True)
print(f"Best epoch {state['best_epoch']} (validation loss {state['best_val_loss']:.4f}) | training time {state['train_seconds'] / 60:.1f} min")

In [ ]:
# Cell 14 — Validation summaries with the best model: all chunks vs first chunk only (long-document comparison), and the
# share of each chunk summary to keep (length control). The best keep-fraction is used for the test summaries.
best = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(best["trainable"], strict=False)
refs = [d["summary"] for d in data["val"]]
start_time = time.time()
val_chunk_summaries = summarise(data["val"])
gen_minutes = (time.time() - start_time) / 60
results = {}
for keep in KEEP_FRACTIONS:
    results[f"all_chunks_keep_{keep}"] = score_summaries(refs, [join_summary(c, keep) for c in val_chunk_summaries])
results["first_chunk_only"] = score_summaries(refs, [join_summary(c) for c in summarise(data["val"], first_chunk_only=True)])
for k, v in results.items():
    print(f"  {k:24s}: {describe(v)}")
KEEP = max(KEEP_FRACTIONS, key=lambda k: results[f"all_chunks_keep_{k}"]["rouge1"] + results[f"all_chunks_keep_{k}"]["rougeL"])
print(f"Chosen keep-fraction {KEEP} | validation generation {gen_minutes:.1f} min for {len(data['val'])} documents")
example = data["val"][0]
print("\nExample (validation) — reference:\n", example["summary"][:700], "\n\nGenerated:\n", join_summary(val_chunk_summaries[0], KEEP)[:700])
json.dump({"best_epoch": best["epoch"], "history": state["history"], "val_loss_phase1": state.get("val_loss_phase1"),
           "validation": results, "keep_fraction": KEEP}, open(OUT_DIR / "val_metrics.json", "w"), indent=2)

In [ ]:
# Cell 15 — Test summaries (all chunks, chosen keep-fraction), format checks and submission.zip.
# prediction.json is a JSON list of {"id": ..., "summary": "..."} with exactly one non-empty summary per test document.
start_time = time.time()
test_chunk_summaries = summarise(data["test"])
test_minutes = (time.time() - start_time) / 60
prediction = [{"id": d["id"], "summary": join_summary(c, KEEP)} for d, c in zip(data["test"], test_chunk_summaries)]
for p, d in zip(prediction, data["test"]):
    if not p["summary"].strip():                                  # never submit an empty summary
        p["summary"] = tokenizer.decode(d["chunks"][0][:200].tolist())
assert [p["id"] for p in prediction] == [d["id"] for d in test_docs] and len({p["id"] for p in prediction}) == len(prediction)
assert all(isinstance(p["summary"], str) and p["summary"].strip() for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"), indent=1)
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
words = np.array([len(p["summary"].split()) for p in prediction])
print(f"submission.zip: {len(prediction)} summaries | words per summary median {int(np.median(words))} "
      f"(training references median {int(np.median([len(d['summary'].split()) for d in train_docs]))}) | generation {test_minutes:.1f} min")

In [ ]:
# Cell 16 — Run summary for the README, the experiment log and results/submissions.csv.
summary = {
    "run": RUN_NAME, "backbone": "Architecture A — decoder-only Transformer (Phase 1)",
    "technique": "LORMA (attention + FFN), prefix-LM chunk-level summarization, joined with redundancy removal",
    "lorma_rank": LORMA_RANK, "trainable_params": n_trainable, "total_params": n_total,
    "train_pairs": len(train_pairs), "val_docs": len(data["val"]), "epochs": state["epoch"], "best_epoch": state["best_epoch"],
    "val_loss_phase1": state.get("val_loss_phase1"), "val_loss_best": state["best_val_loss"],
    "train_minutes": round(state["train_seconds"] / 60, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "validation": results, "keep_fraction": KEEP, "test_generation_minutes": round(test_minutes, 1),
    "test_words_median": int(np.median(words)),
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))